# CHURNGUARD AI
## Customer Churn Prediction & Retention Intelligence System

**Domain:** Artificial Intelligence & Machine Learning  
**Language:** Python  
**Environment:** Google Colab  
**Problem Type:** Supervised Machine Learning – Binary Classification

### Project Goal
Build an end-to-end machine-learning system that analyzes historical customer information and predicts whether a customer is likely to churn. The workflow is:

**Raw Data → Data Understanding → Cleaning → Feature Preparation → Model Training → Evaluation → Prediction → Business Insight**

The project follows the assigned requirements: data cleaning, categorical/numerical preprocessing, at least two classification models, model comparison, evaluation with accuracy/precision/recall/F1-score/confusion matrix, feature importance, probability-based risk output, model saving, improvement, and business interpretation.

## Step 1 — Install / Import Libraries
Run this notebook from top to bottom in Google Colab.

In [ ]:
!pip -q install joblib reportlab

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    RocCurveDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

print("Libraries imported successfully.")

## Step 2 — Understand the Problem

### Q1. What is customer churn?
Customer churn means a customer stops using a company's product or service.

### Q2. Why would a company want to predict churn?
1. To identify customers who may leave.
2. To plan targeted retention campaigns.
3. To reduce revenue loss and improve customer relationships.

### Q3. What is the target variable?
The target variable is **Churn**, which indicates whether a customer left the service.

### Q4. Is this classification or regression?
This is a **binary classification** problem because the model predicts one of two outcomes: churn or no churn.

### Q5. What is a false negative?
A false negative occurs when the model predicts that a customer will stay, but the customer actually churns. In a retention setting, this can mean a potentially at-risk customer is missed by the retention team.

## Step 3 — Load the IBM Telco Customer Churn Dataset

The assigned brief recommends the IBM Telco Customer Churn dataset. The original IBM sample contains 7,043 customer records and 21 columns, with `Churn` as the outcome field.

The code below downloads the CSV directly in Colab. If your institution/MainCrafts provides an official CSV, upload it to Colab and set `LOCAL_FILE` to its filename instead.

In [ ]:
LOCAL_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
IBM_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if os.path.exists(LOCAL_FILE):
    data = pd.read_csv(LOCAL_FILE)
    print("Loaded local dataset:", LOCAL_FILE)
else:
    data = pd.read_csv(IBM_URL)
    print("Loaded IBM dataset from the public source.")

print("Dataset loaded successfully.")

## Step 4 — Dataset Overview

In [ ]:
print("First 5 rows:")
display(data.head())

print("\nShape:", data.shape)

print("\nData types and non-null counts:")
data.info()

print("\nDescriptive statistics:")
display(data.describe(include="all").T)

print("\nMissing values:")
display(data.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("\nDuplicate rows:", data.duplicated().sum())

print("\nDuplicate customer IDs:", data["customerID"].duplicated().sum())

print("\nTarget distribution:")
display(data["Churn"].value_counts())
display((data["Churn"].value_counts(normalize=True)*100).round(2).to_frame("percentage"))

## Step 5 — Data Cleaning

In [ ]:
df = data.copy()

# TotalCharges contains blank strings in the raw dataset; convert to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove exact duplicate rows.
df = df.drop_duplicates().copy()

# Remove duplicate customer IDs if any remain.
df = df.drop_duplicates(subset=["customerID"]).copy()

# Target encoding: No = 0, Yes = 1.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("After cleaning:")
print("Shape:", df.shape)
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Duplicate rows:", df.duplicated().sum())

## Step 6 — Exploratory Data Analysis (EDA)

The following charts examine churn distribution and important customer characteristics. EDA is used to understand the data before training the model.

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()

plt.figure(figsize=(8,5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(data=df, x="tenure", hue="Churn", kde=True, bins=30, element="step")
plt.title("Tenure Distribution by Churn")
plt.xlabel("Tenure (months)")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Monthly Charges")
plt.show()

### Correlation Heatmap

In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number).copy()

plt.figure(figsize=(10,7))
sns.heatmap(numeric_for_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## Step 7 — Feature Preparation

`customerID` is an identifier, not a useful predictive feature, so it is excluded.

The remaining columns are separated into:
- numerical features
- categorical features

A preprocessing pipeline is used so that imputation, scaling and one-hot encoding are performed consistently for training and test data.

In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)
print("\nTotal features before encoding:", X.shape[1])

## Step 8 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Training churn rate:", round(y_train.mean()*100, 2), "%")
print("Testing churn rate:", round(y_test.mean()*100, 2), "%")

## Step 9 — Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created.")

## Step 10 — Model 1: Logistic Regression

Logistic Regression is a standard classification algorithm that estimates the probability of a binary outcome.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")

## Step 11 — Model 2: Random Forest

Random Forest is an ensemble classification method that combines many decision trees.

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully.")

## Step 12 — Evaluate Both Models

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob)
])

display(results.round(4))

### Detailed Classification Reports

In [ ]:
print("LOGISTIC REGRESSION")
print(classification_report(y_test, logistic_pred, target_names=["No Churn", "Churn"]))

print("\nRANDOM FOREST")
print(classification_report(y_test, rf_pred, target_names=["No Churn", "Churn"]))

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm1 = confusion_matrix(y_test, logistic_pred)
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Logistic Regression - Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

cm2 = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm2, annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title("Random Forest - Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

### ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8,6))
RocCurveDisplay.from_predictions(y_test, logistic_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
plt.title("ROC Curve Comparison")
plt.show()

## Step 13 — Model Improvement

The project requires at least one meaningful improvement. Here we use class weighting for Logistic Regression to give additional importance to the churn class, which can help address class imbalance.

In [ ]:
improved_logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

improved_logistic_model.fit(X_train, y_train)

improved_pred = improved_logistic_model.predict(X_test)
improved_prob = improved_logistic_model.predict_proba(X_test)[:, 1]

before_f1 = f1_score(y_test, logistic_pred, zero_division=0)
after_f1 = f1_score(y_test, improved_pred, zero_division=0)

print("Before improvement - Logistic Regression F1:", round(before_f1, 4))
print("After improvement  - Balanced Logistic F1:", round(after_f1, 4))

improvement_results = pd.DataFrame([
    evaluate_model("Logistic Regression - Before", y_test, logistic_pred, logistic_prob),
    evaluate_model("Logistic Regression - After", y_test, improved_pred, improved_prob)
])

display(improvement_results.round(4))

## Step 14 — Select a Model

For this project, the final selection is based primarily on **F1-score**, while also considering precision, recall and ROC-AUC. The code below selects the model with the highest F1-score among the tested models.

This is a transparent, reproducible selection rule rather than a claim that one metric is universally best for every business.

In [ ]:
candidate_models = {
    "Logistic Regression": (logistic_model, logistic_pred, logistic_prob),
    "Random Forest": (rf_model, rf_pred, rf_prob),
    "Balanced Logistic Regression": (improved_logistic_model, improved_pred, improved_prob)
}

selection_rows = []
for name, (_, pred, prob) in candidate_models.items():
    selection_rows.append(evaluate_model(name, y_test, pred, prob))

selection_table = pd.DataFrame(selection_rows).sort_values(
    by=["F1 Score", "Recall", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

display(selection_table.round(4))

selected_name = selection_table.loc[0, "Model"]
selected_model, selected_pred, selected_prob = candidate_models[selected_name]

print("Selected model:", selected_name)

## Step 15 — Feature Importance

In [ ]:
# Permutation importance works directly with the full preprocessing + model pipeline.
perm = permutation_importance(
    selected_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="f1",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(9,6))
top_imp = importance_df.head(12).sort_values("Importance")
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.title(f"Top Features by Permutation Importance — {selected_name}")
plt.xlabel("Mean decrease in F1 after permutation")
plt.show()

## Step 16 — Prediction Probability and Risk Category

The model probability is converted into three simple business-oriented risk categories:
- **HIGH:** probability ≥ 0.70
- **MEDIUM:** 0.40 to < 0.70
- **LOW:** < 0.40

These thresholds are project rules for demonstration and should be calibrated using business costs and validation data before production use.

In [ ]:
def risk_category(probability):
    if probability >= 0.70:
        return "HIGH"
    elif probability >= 0.40:
        return "MEDIUM"
    return "LOW"

risk_counts = pd.Series(selected_prob).apply(risk_category).value_counts()
display(risk_counts.to_frame("customers"))

## Step 17 — Predict Churn Risk for a New Customer

In [ ]:
# Example customer. Change these values to test other customers.
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 5,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.0,
    "TotalCharges": 425.0
}])

new_probability = selected_model.predict_proba(new_customer)[0, 1]
new_prediction = selected_model.predict(new_customer)[0]
new_risk = risk_category(new_probability)

print("CHURNGUARD AI — NEW CUSTOMER PREDICTION")
print("----------------------------------------")
print("Predicted churn:", "YES" if new_prediction == 1 else "NO")
print("Churn probability:", f"{new_probability:.2%}")
print("Customer Churn Risk:", new_risk)

## Step 18 — Save the Trained Model

The trained pipeline includes preprocessing and the classifier, so the same transformations are applied when the saved model is loaded later.

In [ ]:
MODEL_FILE = "churn_guard_ai_model.joblib"
joblib.dump(selected_model, MODEL_FILE)

print("Saved:", MODEL_FILE)
print("File exists:", os.path.exists(MODEL_FILE))

## Step 19 — Final Results

The assigned brief requires the notebook to finish with a project summary containing the dataset, number of records, models tested, selected model, evaluation metrics, important findings, business recommendation and limitations.

In [ ]:
selected_metrics = selection_table[selection_table["Model"] == selected_name].iloc[0]

print("="*65)
print("CHURNGUARD AI — FINAL RESULTS")
print("="*65)
print("Dataset: IBM Telco Customer Churn")
print("Number of records:", len(df))
print("Models tested: Logistic Regression, Random Forest, Balanced Logistic Regression")
print("Selected model:", selected_name)
print(f"Accuracy:  {selected_metrics['Accuracy']:.4f}")
print(f"Precision: {selected_metrics['Precision']:.4f}")
print(f"Recall:    {selected_metrics['Recall']:.4f}")
print(f"F1 Score:  {selected_metrics['F1 Score']:.4f}")
print(f"ROC-AUC:   {selected_metrics['ROC-AUC']:.4f}")
print("\nMost important findings:")
for i, row in importance_df.head(3).reset_index(drop=True).iterrows():
    print(f"{i+1}. {row['Feature']}")

print("\nBusiness recommendation:")
print("Customers identified as higher-risk can be considered for retention campaigns, service reviews, or targeted offers.")

print("\nLimitations:")
print("Model predictions are not guarantees that an individual customer will churn.")
print("The dataset is an educational sample and should be validated on current company data before production use.")
print("="*65)

## Step 20 — Automatically Generate a Project Report PDF

Run this final cell after all previous cells. It creates a submission-ready report using the actual metrics produced by your notebook.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER

report_file = "CHURNGUARD_AI_Project_Report.pdf"

styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    "TitleCustom", parent=styles["Title"], alignment=TA_CENTER, fontSize=20, spaceAfter=18
)
heading_style = styles["Heading2"]
body_style = styles["BodyText"]

doc = SimpleDocTemplate(report_file, pagesize=A4, rightMargin=45, leftMargin=45, topMargin=45, bottomMargin=45)
story = []

story.append(Paragraph("CHURNGUARD AI", title_style))
story.append(Paragraph("Customer Churn Prediction & Retention Intelligence System", title_style))
story.append(Spacer(1, 10))

sections = [
    ("1. Problem Statement",
     "Businesses can lose customers for many reasons. The objective of CHURNGUARD AI is to learn patterns from historical customer information and predict whether a customer is likely to churn."),
    ("2. Objective",
     "Build an end-to-end binary classification system covering data understanding, cleaning, preprocessing, model training, evaluation, prediction, feature interpretation and business recommendations."),
    ("3. Dataset Description",
     f"The IBM Telco Customer Churn sample dataset was used. The cleaned dataset contains {len(df):,} records. The target variable is Churn, encoded as 0 for No and 1 for Yes."),
    ("4. Technologies Used",
     "Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Joblib and ReportLab. The recommended execution environment is Google Colab."),
    ("5. Methodology",
     "The workflow includes dataset loading, data inspection, duplicate checking, TotalCharges conversion, target encoding, train/test splitting, numerical scaling, categorical one-hot encoding, model training, evaluation, improvement, feature importance and risk categorization."),
    ("6. Models Tested",
     "Logistic Regression, Random Forest and an improved Logistic Regression using class_weight='balanced'."),
    ("7. Model Evaluation",
     f"The selected model was {selected_name}. Accuracy = {selected_metrics['Accuracy']:.4f}; Precision = {selected_metrics['Precision']:.4f}; Recall = {selected_metrics['Recall']:.4f}; F1-score = {selected_metrics['F1 Score']:.4f}; ROC-AUC = {selected_metrics['ROC-AUC']:.4f}."),
    ("8. Important Findings",
     "The three highest-ranked raw input features by permutation importance are: " +
     ", ".join(importance_df.head(3)["Feature"].tolist()) + ". These are model-associated signals, not proof of causation."),
    ("9. Business Interpretation",
     "Customers identified as higher-risk could be considered for retention campaigns, service reviews or targeted offers. The model can support prioritization, but human/business review remains necessary."),
    ("10. Risk Output",
     f"The demonstration converts churn probability into LOW (<40%), MEDIUM (40%–69.99%) and HIGH (≥70%) risk categories."),
    ("11. Limitations",
     "Predictions are not guarantees. The sample dataset is intended for educational/analytical use. Production deployment would require current company data, validation, monitoring, threshold calibration, privacy review and business-cost analysis."),
    ("12. Future Improvements",
     "Hyperparameter tuning, threshold optimization using retention costs, probability calibration, additional feature engineering, temporal validation with time-based data, model monitoring and a Streamlit interface.")
]

for heading, text_ in sections:
    story.append(Paragraph(heading, heading_style))
    story.append(Paragraph(text_, body_style))
    story.append(Spacer(1, 8))

story.append(Spacer(1, 8))
story.append(Paragraph("Final Model Metrics", heading_style))
table_data = [
    ["Metric", "Value"],
    ["Accuracy", f"{selected_metrics['Accuracy']:.4f}"],
    ["Precision", f"{selected_metrics['Precision']:.4f}"],
    ["Recall", f"{selected_metrics['Recall']:.4f}"],
    ["F1 Score", f"{selected_metrics['F1 Score']:.4f}"],
    ["ROC-AUC", f"{selected_metrics['ROC-AUC']:.4f}"],
]
tbl = Table(table_data, colWidths=[180, 180])
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.5, colors.grey),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("ALIGN", (1,1), (1,-1), "CENTER"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ("TOPPADDING", (0,0), (-1,-1), 6),
    ("BOTTOMPADDING", (0,0), (-1,-1), 6),
]))
story.append(tbl)

doc.build(story)

print("Created:", report_file)

# Submission Checklist

Before submitting, make sure you have:

- [ ] `CHURNGUARD_AI_Complete_Project.ipynb`
- [ ] `churn_guard_ai_model.joblib`
- [ ] `CHURNGUARD_AI_Project_Report.pdf`
- [ ] Dataset CSV or a documented dataset source
- [ ] Dataset preview
- [ ] EDA charts
- [ ] Correlation heatmap
- [ ] Model comparison
- [ ] Confusion matrix
- [ ] Final prediction
- [ ] Final results
- [ ] README
- [ ] GitHub repository link
- [ ] Google Colab notebook link
- [ ] Published LinkedIn post link, if required by your submission form

### Important
Run every notebook cell before downloading/submitting the report. The final PDF is generated from the actual model results produced by your run.